In [3]:
import json
import os
import copy
import warnings
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from tqdm import tqdm
warnings.filterwarnings("ignore")

In [ ]:
# %%
"""
task2_benchmark.py — Ball Count Regression Benchmark

Transfer learning flow per model:
  1. Pre-train on auxiliary datasets (1, 2, 3)
  2. 5-fold CV on original dataset (247 images, Dot excluded)
  3. Final training on all original data — saves best .pth per model

Models:
  1. EfficientNet-B0                   MSE loss
  2. MobileNetV3-Small frozen          Poisson loss
  3. MobileNetV3-Small 5ch             Poisson loss
  4. DINOv2 ViT-S/14 frozen + MLP     Poisson loss
  5. DINOv2 ViT-S/14 2 unfrozen       Poisson loss
  6. DINOv2 ViT-S/14 4 unfrozen       Poisson loss
  7. DINOv2 ViT-B/14 2 unfrozen       Poisson loss
"""

# %%
# ============================================================
# IMPORTS
# ============================================================
import json
import os
import copy
import warnings
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from tqdm import tqdm
warnings.filterwarnings("ignore")

# %%
# ============================================================
# GLOBAL CONFIG
# ============================================================

# --- paths ---
UNIFIED_JSON   = "unified_dataset.json"
MODELS_DIR     = "models"
RESULTS_FILE   = "task2_results.txt"

# --- dataset ---
ORIGINAL_DS_ID = 4   # dataset 4 = original; Dots already excluded by build_dataset_json.py

# --- model ---
IMG_SIZE           = 224
BATCH_SIZE         = 8
SEED               = 42

# --- pre-training (auxiliary datasets) ---
PRETRAIN_EPOCHS    = 5 
PRETRAIN_LR        = 1e-3
PRETRAIN_WD        = 1e-2
PRETRAIN_PATIENCE  = 8

# --- fine-tuning (original dataset) ---
FINETUNE_EPOCHS    = 100
FINETUNE_LR        = 1e-3
FINETUNE_WD        = 1e-2
FINETUNE_PATIENCE  = 10

# --- cross-validation ---
N_FOLDS            = 5

# --- final training ---
FINAL_VAL_SPLIT    = 0.1   # small hold-out for early stopping in final training

# ---- setup ----
os.makedirs(MODELS_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# %%
# ============================================================
# DATA LOADING
# ============================================================

def load_data():
    """
    Load unified_dataset.json.
    Dots are already excluded from dataset 4 by build_dataset_json.py.
    Returns:
      aux_data  — list of dicts for auxiliary datasets (1, 2, 3)
      orig_data — list of dicts for original dataset (4)
    """
    with open(UNIFIED_JSON) as f:
        entries = json.load(f)

    aux_data, orig_data = [], []

    for e in entries:
        if e["dataset"] == ORIGINAL_DS_ID:
            orig_data.append({"image_path": e["image_path"], "total_balls": e["total_balls"]})
        else:
            aux_data.append({"image_path": e["image_path"], "total_balls": e["total_balls"]})

    print(f"Auxiliary data : {len(aux_data)} images")
    print(f"Original data  : {len(orig_data)} images")
    arr = [d["total_balls"] for d in orig_data]
    print(f"  Ball count   : min={min(arr)}  max={max(arr)}  mean={np.mean(arr):.1f}  std={np.std(arr):.1f}")

    return aux_data, orig_data


aux_data, orig_data = load_data()

# %%
# ============================================================
# TABLE SEGMENTATION + PREPROCESSING
# ============================================================

def order_corners(corners):
    pts = np.array(corners, dtype=np.float32)
    s   = pts.sum(axis=1)
    d   = np.diff(pts, axis=1).ravel()
    return np.array([
        pts[np.argmin(s)], pts[np.argmin(d)],
        pts[np.argmax(s)], pts[np.argmax(d)],
    ], dtype=np.float32)


def segment_table(img, erode_px=20):
    hsv        = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    blue_mask  = cv2.inRange(hsv, np.array([90, 80, 50]),  np.array([130, 255, 255]))
    green_mask = cv2.inRange(hsv, np.array([35, 80, 50]),  np.array([85,  255, 255]))
    mask       = cv2.bitwise_or(blue_mask, green_mask)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return np.ones(img.shape[:2], dtype=np.uint8) * 255, None

    tc      = max(contours, key=cv2.contourArea)
    approx  = cv2.approxPolyDP(tc, 0.02 * cv2.arcLength(tc, True), True)
    corners = (approx.reshape(4, 2).astype(np.float32) if len(approx) == 4
               else cv2.boxPoints(cv2.minAreaRect(tc)).astype(np.float32))
    corners    = order_corners(corners)
    clean_mask = np.zeros(img.shape[:2], dtype=np.uint8)
    cv2.fillPoly(clean_mask, [corners.astype(np.int32)], 255)

    if erode_px > 0:
        k          = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (erode_px * 2, erode_px * 2))
        clean_mask = cv2.erode(clean_mask, k, iterations=1)

    return clean_mask, corners


def apply_table_mask(img):
    mask, _ = segment_table(img)
    return cv2.bitwise_and(img, img, mask=mask)


def build_5channel(img):
    """
    Build (H, W, 5) float32 tensor:
      ch 0-2: RGB masked
      ch 3:   diff map (euclidean distance to cloth mean)
      ch 4:   gray_enhanced (diff + saturation CLAHE + gray)
    All channels normalized to [0, 1].
    """
    mask, _  = segment_table(img)
    masked   = cv2.bitwise_and(img, img, mask=mask)

    h, s, v  = cv2.split(cv2.cvtColor(masked, cv2.COLOR_BGR2HSV))
    boosted  = cv2.cvtColor(
        cv2.merge([h,
                   cv2.convertScaleAbs(s, alpha=1.35, beta=10),
                   cv2.convertScaleAbs(v, alpha=1.15, beta=8)]),
        cv2.COLOR_HSV2BGR
    )

    cloth_mean    = np.array(cv2.mean(boosted, mask=mask)[:3], dtype=np.float32)
    diff          = np.clip(
        np.sqrt(np.sum((boosted.astype(np.float32) - cloth_mean) ** 2, axis=2)),
        0, 255
    ).astype(np.uint8)

    _, s2, _      = cv2.split(cv2.cvtColor(boosted, cv2.COLOR_BGR2HSV))
    clahe         = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_raw      = cv2.cvtColor(boosted, cv2.COLOR_BGR2GRAY)
    gray_enhanced = cv2.addWeighted(diff, 0.8, clahe.apply(s2), 0.2, 0)
    gray_enhanced = cv2.GaussianBlur(
        cv2.addWeighted(gray_enhanced, 0.85, clahe.apply(gray_raw), 0.15, 0),
        (3, 3), 2
    )

    rgb = cv2.cvtColor(boosted, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return np.concatenate([
        rgb,
        (diff.astype(np.float32) / 255.0)[:, :, None],
        (gray_enhanced.astype(np.float32) / 255.0)[:, :, None],
    ], axis=2)   # (H, W, 5)


# %%
# ============================================================
# DATASETS AND DATALOADERS
# ============================================================

_train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
_val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
_MEAN_5CH = torch.tensor([0.485, 0.456, 0.406, 0.5, 0.5]).view(5, 1, 1)
_STD_5CH  = torch.tensor([0.229, 0.224, 0.225, 0.5, 0.5]).view(5, 1, 1)


class BallDataset(Dataset):
    def __init__(self, samples, augment=False):
        self.samples = samples
        self.tf      = _train_tf if augment else _val_tf

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s    = self.samples[idx]
        img  = cv2.imread(s["image_path"])
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(apply_table_mask(img), cv2.COLOR_BGR2RGB)
        return self.tf(Image.fromarray(img)), torch.tensor(float(s["total_balls"]))


class BallDataset5Ch(Dataset):
    def __init__(self, samples, augment=False):
        self.samples = samples
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        img = cv2.imread(s["image_path"])
        t   = (build_5channel(img) if img is not None
               else np.zeros((IMG_SIZE, IMG_SIZE, 5), dtype=np.float32))
        t   = cv2.resize(t, (IMG_SIZE, IMG_SIZE))

        if self.augment:
            if np.random.rand() > 0.5:
                t = np.fliplr(t).copy()
            if np.random.rand() > 0.5:
                t = np.flipud(t).copy()

        tensor = (torch.from_numpy(t.transpose(2, 0, 1)) - _MEAN_5CH) / _STD_5CH
        return tensor, torch.tensor(float(s["total_balls"]))


def make_loader(samples, augment=False, five_channel=False, bs=BATCH_SIZE):
    cls = BallDataset5Ch if five_channel else BallDataset
    return DataLoader(cls(samples, augment=augment),
                      batch_size=bs, shuffle=augment, num_workers=0, pin_memory=True)


# %%
# ============================================================
# MODEL DEFINITIONS
# ============================================================

def _reg_head(in_features, softplus=True):
    layers = [
        nn.Linear(in_features, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(128, 32), nn.ReLU(),
        nn.Linear(32, 1),
    ]
    if softplus:
        layers.append(nn.Softplus())
    return nn.Sequential(*layers)


class EfficientNetRegressor(nn.Module):
    """EfficientNet-B0 with regression head — MSE loss."""
    def __init__(self):
        super().__init__()
        bb               = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_f             = bb.classifier[1].in_features
        bb.classifier    = nn.Identity()
        self.backbone    = bb
        self.head        = _reg_head(in_f, softplus=False)

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(1)


class MobileNetFrozenRegressor(nn.Module):
    """MobileNetV3-Small fully frozen backbone — Poisson loss."""
    def __init__(self):
        super().__init__()
        bb = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        for p in bb.parameters():
            p.requires_grad = False
        self.features = bb.features
        self.avgpool  = bb.avgpool
        self.head     = _reg_head(576)

    def forward(self, x):
        with torch.no_grad():
            x = self.avgpool(self.features(x)).flatten(1)
        return self.head(x).squeeze(1)


class MobileNet5ChRegressor(nn.Module):
    """MobileNetV3-Small with 5-channel input, last 2 backbone layers trainable — Poisson loss."""
    def __init__(self):
        super().__init__()
        bb      = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        old     = bb.features[0][0]
        new     = nn.Conv2d(5, old.out_channels, old.kernel_size,
                            stride=old.stride, padding=old.padding, bias=False)
        with torch.no_grad():
            new.weight[:, :3] = old.weight
            new.weight[:, 3:] = old.weight[:, :2].mean(dim=1, keepdim=True)
        bb.features[0][0] = new

        # Freeze all layers except last 2
        for layer in list(bb.features.children())[:-2]:
            for p in layer.parameters():
                p.requires_grad = False

        self.features = bb.features
        self.avgpool  = bb.avgpool
        self.head     = _reg_head(576)

    def forward(self, x):
        return self.head(self.avgpool(self.features(x)).flatten(1)).squeeze(1)


class DinoRegressor(nn.Module):
    """DINOv2 backbone + MLP head. Supports 0-N unfrozen transformer blocks — Poisson loss."""
    def __init__(self, backbone_name="dinov2_vits14", unfreeze_blocks=0):
        super().__init__()
        self.backbone = torch.hub.load(
            "facebookresearch/dinov2", backbone_name, verbose=False
        )
        for p in self.backbone.parameters():
            p.requires_grad = False

        for block in self.backbone.blocks[-unfreeze_blocks:] if unfreeze_blocks > 0 else []:
            for p in block.parameters():
                p.requires_grad = True
        if unfreeze_blocks > 0:
            for p in self.backbone.norm.parameters():
                p.requires_grad = True

        self.head = _reg_head(self.backbone.embed_dim)

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(1)


# ── Model registry ────────────────────────────────────────────────────────────
MODEL_CONFIGS = [
    {"name": "efficientnet_b0",      "build": lambda: EfficientNetRegressor(),                         "loss": "mse",     "5ch": False},
    {"name": "mobilenet_frozen",     "build": lambda: MobileNetFrozenRegressor(),                      "loss": "poisson", "5ch": False},
    {"name": "mobilenet_5ch",        "build": lambda: MobileNet5ChRegressor(),                         "loss": "poisson", "5ch": True },
    {"name": "dino_vits14_frozen",   "build": lambda: DinoRegressor("dinov2_vits14", 0),               "loss": "poisson", "5ch": False},
    {"name": "dino_vits14_2blocks",  "build": lambda: DinoRegressor("dinov2_vits14", 2),               "loss": "poisson", "5ch": False},
    {"name": "dino_vits14_4blocks",  "build": lambda: DinoRegressor("dinov2_vits14", 4),               "loss": "poisson", "5ch": False},
    {"name": "dino_vitb14_2blocks",  "build": lambda: DinoRegressor("dinov2_vitb14", 2),               "loss": "poisson", "5ch": False},
]

# %%
# ============================================================
# TRAINING UTILITIES
# ============================================================

def get_criterion(loss_type):
    if loss_type == "mse":
        return nn.MSELoss()
    return nn.PoissonNLLLoss(log_input=False, full=True, reduction="mean")


def get_optimizer(model, lr, wd):
    """
    Differential LRs: head at lr, trainable backbone layers at lr/10.
    """
    head_ids        = {id(p) for p in model.head.parameters()} if hasattr(model, "head") else set()
    backbone_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    head_params     = [p for p in model.parameters() if p.requires_grad and id(p) in head_ids]

    groups = [{"params": head_params, "lr": lr}]
    if backbone_params:
        groups.append({"params": backbone_params, "lr": lr * 0.1})

    return torch.optim.AdamW(groups, weight_decay=wd)


def train_one_epoch(model, loader, optimizer, criterion, device, desc="train"):
    model.train()
    total = 0.0
    bar   = tqdm(loader, desc=desc, leave=False, ncols=80)
    for imgs, labels in bar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(imgs)
        bar.set_postfix(loss=f"{loss.item():.4f}")
    return total / len(loader.dataset)


@torch.no_grad()
def eval_epoch(model, loader, device, desc="val"):
    model.eval()
    preds_all, targets_all = [], []
    for imgs, labels in tqdm(loader, desc=desc, leave=False, ncols=80):
        preds_all.extend(model(imgs.to(device)).cpu().numpy())
        targets_all.extend(labels.numpy())
    p = np.array(preds_all,   dtype=np.float32)
    t = np.array(targets_all, dtype=np.float32)
    return float(np.mean(np.abs(p - t))), float(np.mean((p - t) ** 2))


def train_model(model, train_dl, val_dl, criterion, optimizer,
                n_epochs, patience, device, tag=""):
    """
    Train with ReduceLROnPlateau + early stopping.
    Returns (model with best weights loaded, best_mae).
    """
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=patience // 2, factor=0.5
    )
    best_mae   = float("inf")
    best_state = None
    no_improve = 0

    epoch_bar = tqdm(range(1, n_epochs + 1), desc=f"{tag}epochs", ncols=90)
    for epoch in epoch_bar:
        tl          = train_one_epoch(model, train_dl, optimizer, criterion, device, desc="  batch")
        val_mae, _  = eval_epoch(model, val_dl, device, desc="  val")
        scheduler.step(val_mae)

        if val_mae < best_mae:
            best_mae   = val_mae
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        epoch_bar.set_postfix(val_mae=f"{val_mae:.3f}", best=f"{best_mae:.3f}")

        if no_improve >= patience:
            tqdm.write(f"  {tag}Early stop epoch {epoch} — best MAE={best_mae:.3f}")
            break

    model.load_state_dict(best_state)
    return model, best_mae


# %%
# ============================================================
# PHASE 1 — PRE-TRAINING ON AUXILIARY DATASETS
# ============================================================

def pretrain(cfg, aux_data, device):
    name = cfg["name"]
    print(f"\n{'='*55}\n[Pre-train] {name}\n{'='*55}")

    rng      = np.random.RandomState(SEED)
    idx      = rng.permutation(len(aux_data))
    val_n    = max(1, int(0.1 * len(aux_data)))
    train_s  = [aux_data[i] for i in idx[val_n:]]
    val_s    = [aux_data[i] for i in idx[:val_n]]

    model     = cfg["build"]().to(device)
    criterion = get_criterion(cfg["loss"])
    optimizer = get_optimizer(model, PRETRAIN_LR, PRETRAIN_WD)

    train_dl = make_loader(train_s, augment=True,  five_channel=cfg["5ch"])
    val_dl   = make_loader(val_s,   augment=False, five_channel=cfg["5ch"])

    model, best_mae = train_model(
        model, train_dl, val_dl, criterion, optimizer,
        PRETRAIN_EPOCHS, PRETRAIN_PATIENCE, device, tag="[pretrain] "
    )

    save_path = os.path.join(MODELS_DIR, f"{name}_pretrained.pth")
    torch.save(model.state_dict(), save_path)
    print(f"  Saved pre-trained weights → {save_path}  (Val MAE={best_mae:.3f})")

    return model.state_dict()


pretrained_states = {}
for cfg in tqdm(MODEL_CONFIGS, desc="pre-training models", ncols=90):
    pretrained_states[cfg["name"]] = pretrain(cfg, aux_data, device)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# %%
# ============================================================
# PHASE 2 — 5-FOLD CV ON ORIGINAL DATASET
# ============================================================

def run_kfold(cfg, orig_data, pretrained_state, device):
    name = cfg["name"]
    print(f"\n{'='*55}\n[5-Fold CV] {name}\n{'='*55}")

    indices  = np.random.RandomState(SEED).permutation(len(orig_data))
    folds    = np.array_split(indices, N_FOLDS)
    results  = []

    for fold_idx, test_idx in enumerate(folds):
        train_idx = np.concatenate([f for i, f in enumerate(folds) if i != fold_idx])
        train_s   = [orig_data[i] for i in train_idx]
        test_s    = [orig_data[i] for i in test_idx]

        # Small internal val for early stopping (last 10% of train)
        val_n   = max(1, int(0.1 * len(train_s)))
        val_s   = train_s[-val_n:]
        train_s = train_s[:-val_n]

        model = cfg["build"]().to(device)
        model.load_state_dict(pretrained_state, strict=False)

        criterion = get_criterion(cfg["loss"])
        optimizer = get_optimizer(model, FINETUNE_LR, FINETUNE_WD)

        train_dl = make_loader(train_s, augment=True,  five_channel=cfg["5ch"])
        val_dl   = make_loader(val_s,   augment=False, five_channel=cfg["5ch"])
        test_dl  = make_loader(test_s,  augment=False, five_channel=cfg["5ch"])

        model, _ = train_model(
            model, train_dl, val_dl, criterion, optimizer,
            FINETUNE_EPOCHS, FINETUNE_PATIENCE, device,
            tag=f"[fold {fold_idx+1}] "
        )

        mae, mse = eval_epoch(model, test_dl, device)
        results.append({"mae": mae, "mse": mse})
        print(f"  Fold {fold_idx+1}/{N_FOLDS} → MAE={mae:.3f}  MSE={mse:.3f}")

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    maes = [r["mae"] for r in results]
    mses = [r["mse"] for r in results]
    print(f"\n  CV Summary | MAE: {np.mean(maes):.3f} ± {np.std(maes):.3f} | "
          f"MSE: {np.mean(mses):.3f} ± {np.std(mses):.3f}")

    return results


cv_results = {}
for cfg in MODEL_CONFIGS:
    cv_results[cfg["name"]] = run_kfold(
        cfg, orig_data, pretrained_states[cfg["name"]], device
    )
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# %%
# ============================================================
# PHASE 3 — FINAL TRAINING ON ALL ORIGINAL DATA
# ============================================================

def final_train(cfg, orig_data, pretrained_state, device):
    name = cfg["name"]
    print(f"\n{'='*55}\n[Final Train] {name}\n{'='*55}")

    val_n   = max(1, int(FINAL_VAL_SPLIT * len(orig_data)))
    val_s   = orig_data[-val_n:]
    train_s = orig_data[:-val_n]

    model = cfg["build"]().to(device)
    model.load_state_dict(pretrained_state, strict=False)

    criterion = get_criterion(cfg["loss"])
    optimizer = get_optimizer(model, FINETUNE_LR, FINETUNE_WD)

    train_dl = make_loader(train_s, augment=True,  five_channel=cfg["5ch"])
    val_dl   = make_loader(val_s,   augment=False, five_channel=cfg["5ch"])

    model, best_mae = train_model(
        model, train_dl, val_dl, criterion, optimizer,
        FINETUNE_EPOCHS, FINETUNE_PATIENCE, device, tag="[final] "
    )

    save_path = os.path.join(MODELS_DIR, f"{name}_final.pth")
    torch.save(model.state_dict(), save_path)
    print(f"  Saved → {save_path}  (Val MAE={best_mae:.3f})")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return best_mae


final_maes = {}
for cfg in MODEL_CONFIGS:
    final_maes[cfg["name"]] = final_train(
        cfg, orig_data, pretrained_states[cfg["name"]], device
    )


# %%
# ============================================================
# RESULTS SUMMARY
# ============================================================

header = (
    f"\n{'='*85}\n"
    f"TASK 2 — BALL COUNT REGRESSION  ({N_FOLDS}-fold CV)\n"
    f"{'='*85}\n"
    f"{'Model':<25} | {'CV MAE mean±std':<20} | {'CV MSE mean±std':<20} | Final Val MAE\n"
    f"{'-'*85}"
)

lines = [header]
for cfg in MODEL_CONFIGS:
    n    = cfg["name"]
    maes = [r["mae"] for r in cv_results[n]]
    mses = [r["mse"] for r in cv_results[n]]
    lines.append(
        f"{n:<25} | "
        f"{np.mean(maes):.3f} ± {np.std(maes):.3f}        | "
        f"{np.mean(mses):.3f} ± {np.std(mses):.3f}        | "
        f"{final_maes[n]:.3f}"
    )

lines.append("=" * 85)
summary = "\n".join(lines)
print(summary)

with open(RESULTS_FILE, "w") as f:
    f.write(summary)
print(f"\nResults saved → {RESULTS_FILE}")

# ── Bar chart — CV MAE comparison ─────────────────────────────────────────────
short = [c["name"].replace("dino_", "").replace("mobilenet_", "mob_")
         .replace("efficientnet_", "eff_")
         for c in MODEL_CONFIGS]
means = [np.mean([r["mae"] for r in cv_results[c["name"]]]) for c in MODEL_CONFIGS]
stds  = [np.std( [r["mae"] for r in cv_results[c["name"]]]) for c in MODEL_CONFIGS]

plt.figure(figsize=(12, 5))
plt.bar(short, means, yerr=stds, capsize=6, color="steelblue", alpha=0.85)
plt.ylabel("MAE (balls)")
plt.title(f"Task 2 — {N_FOLDS}-Fold CV MAE by Model")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("task2_cv_comparison.png", dpi=150)
print("Plot saved → task2_cv_comparison.png")


Device: cpu
Auxiliary data : 1045 images
Original data  : 247 images
  Ball count   : min=2  max=16  mean=11.4  std=3.7


pre-training models:   0%|                                          | 0/7 [00:00<?, ?it/s]


[Pre-train] efficientnet_b0


  Saved pre-trained weights → models/efficientnet_b0_pretrained.pth  (Val MAE=1.496)

[Pre-train] mobilenet_frozen


  Saved pre-trained weights → models/mobilenet_frozen_pretrained.pth  (Val MAE=1.771)

[Pre-train] mobilenet_5ch


...

  [final] Early stop epoch 20 — best MAE=1.914
  Saved → models/dino_vits14_2blocks_final.pth  (Val MAE=1.914)

[Final Train] dino_vits14_4blocks


[final] epochs:  23%|██       | 23/100 [08:08<27:14, 21.23s/it, best=1.689, val_mae=1.943]


  [final] Early stop epoch 24 — best MAE=1.689
  Saved → models/dino_vits14_4blocks_final.pth  (Val MAE=1.689)

[Final Train] dino_vitb14_2blocks


[final] epochs:  11%|▉        | 11/100 [05:24<43:48, 29.53s/it, best=1.645, val_mae=2.250]


  [final] Early stop epoch 12 — best MAE=1.645
  Saved → models/dino_vitb14_2blocks_final.pth  (Val MAE=1.645)

TASK 2 — BALL COUNT REGRESSION  (5-fold CV)
Model                     | CV MAE mean±std      | CV MSE mean±std      | Final Val MAE
-------------------------------------------------------------------------------------
efficientnet_b0           | 2.517 ± 0.484        | 10.252 ± 3.639        | 1.842
mobilenet_frozen          | 2.611 ± 0.350        | 10.742 ± 2.768        | 2.237
mobilenet_5ch             | 2.958 ± 0.435        | 13.212 ± 3.010        | 2.249
dino_vits14_frozen        | 2.012 ± 0.145        | 7.227 ± 1.185        | 1.750
dino_vits14_2blocks       | 2.045 ± 0.061        | 6.677 ± 0.419        | 1.914
dino_vits14_4blocks       | 2.222 ± 0.399        | 8.112 ± 2.552        | 1.689
dino_vitb14_2blocks       | 2.082 ± 0.144        | 6.958 ± 0.524        | 1.645

Results saved → task2_results.txt
Plot saved → task2_cv_comparison.png


In [ ]:
# %%
"""
task2_benchmark.py — Ball Count Regression Benchmark

Transfer learning flow per model:
  1. Pre-train on auxiliary datasets (1, 2, 3)
  2. 5-fold CV on original dataset (247 images, Dot excluded)
  3. Final training on all original data — saves best .pth per model

Models:
  1. EfficientNet-B0                   MSE loss
  2. MobileNetV3-Small frozen          Poisson loss
  3. MobileNetV3-Small 5ch             Poisson loss
  4. DINOv2 ViT-S/14 frozen + MLP     Poisson loss
  5. DINOv2 ViT-S/14 2 unfrozen       Poisson loss
  6. DINOv2 ViT-S/14 4 unfrozen       Poisson loss
  7. DINOv2 ViT-B/14 2 unfrozen       Poisson loss
"""

# %%
# ============================================================
# IMPORTS
# ============================================================
import json
import os
import copy
import warnings
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from tqdm import tqdm
warnings.filterwarnings("ignore")

# %%
# ============================================================
# GLOBAL CONFIG
# ============================================================

# --- paths ---
UNIFIED_JSON   = "unified_dataset.json"
MODELS_DIR     = "models"
RESULTS_FILE   = "task2_results.txt"
MASKS_DIR      = "mascaras"

# --- dataset ---
ORIGINAL_DS_ID = 4   # dataset 4 = original; Dots already excluded by build_dataset_json.py
SKIP_DS_ID     = 2   # dataset 2 images are invalid — skip entirely

# --- model ---
IMG_SIZE           = 224
BATCH_SIZE         = 8
SEED               = 42

# --- pre-training (auxiliary datasets) ---
PRETRAIN_EPOCHS    = 30
PRETRAIN_LR        = 1e-3
PRETRAIN_WD        = 1e-2
PRETRAIN_PATIENCE  = 15

# --- fine-tuning (original dataset) ---
FINETUNE_EPOCHS    = 100
FINETUNE_LR        = 1e-3
FINETUNE_WD        = 1e-2
FINETUNE_PATIENCE  = 10

# --- cross-validation ---
N_FOLDS            = 5

# --- final training ---
FINAL_VAL_SPLIT    = 0.1   # small hold-out for early stopping in final training

# ---- setup ----
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(MASKS_DIR,  exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# %%
# ============================================================
# TABLE SEGMENTATION + PREPROCESSING
# ============================================================
def order_corners(corners):
    """Order 4 corners as top-left, top-right, bottom-right, bottom-left."""
    corners = np.array(corners, dtype=np.float32)
    center  = corners.mean(axis=0)
    angles  = np.arctan2(corners[:, 1] - center[1], corners[:, 0] - center[0])
    order   = np.argsort(angles)
    corners = corners[order]
    top_idx = np.argmin(corners[:, 0] + corners[:, 1])
    corners = np.roll(corners, -top_idx, axis=0)
    return corners


def segment_table(img):
    """Detect the billiard table cloth and return (mask, corners)."""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    green_mask = cv2.inRange(hsv, np.array([35, 80, 50]), np.array([95, 255, 255]))
    blue_mask  = cv2.inRange(hsv, np.array([90,  80,  50]), np.array([130, 255, 255]))
    mask = cv2.bitwise_or(blue_mask, green_mask)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        full = np.ones(img.shape[:2], dtype=np.uint8) * 255
        h, w = img.shape[:2]
        corners = np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=np.float32)
        return full, corners

    table_contour = max(contours, key=cv2.contourArea)
    epsilon = 0.02 * cv2.arcLength(table_contour, True)
    approx  = cv2.approxPolyDP(table_contour, epsilon, True)

    if len(approx) == 4:
        corners = approx.reshape(4, 2).astype(np.float32)
    else:
        rect    = cv2.minAreaRect(table_contour)
        corners = cv2.boxPoints(rect).astype(np.float32)

    corners    = order_corners(corners)
    clean_mask = np.zeros(img.shape[:2], dtype=np.uint8)
    cv2.fillPoly(clean_mask, [corners.astype(np.int32)], 255)

    return clean_mask, corners


def apply_table_mask(img):
    mask, _ = segment_table(img)
    return cv2.bitwise_and(img, img, mask=mask)


def build_5channel(img):
    """
    Build (H, W, 5) float32 tensor:
      ch 0-2: RGB masked
      ch 3:   diff map (euclidean distance to cloth mean)
      ch 4:   gray_enhanced (diff + saturation CLAHE + gray)
    All channels normalized to [0, 1].
    """
    mask, _  = segment_table(img)
    masked   = cv2.bitwise_and(img, img, mask=mask)

    h, s, v  = cv2.split(cv2.cvtColor(masked, cv2.COLOR_BGR2HSV))
    boosted  = cv2.cvtColor(
        cv2.merge([h,
                   cv2.convertScaleAbs(s, alpha=1.35, beta=10),
                   cv2.convertScaleAbs(v, alpha=1.15, beta=8)]),
        cv2.COLOR_HSV2BGR
    )

    cloth_mean    = np.array(cv2.mean(boosted, mask=mask)[:3], dtype=np.float32)
    diff          = np.clip(
        np.sqrt(np.sum((boosted.astype(np.float32) - cloth_mean) ** 2, axis=2)),
        0, 255
    ).astype(np.uint8)

    _, s2, _      = cv2.split(cv2.cvtColor(boosted, cv2.COLOR_BGR2HSV))
    clahe         = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_raw      = cv2.cvtColor(boosted, cv2.COLOR_BGR2GRAY)
    gray_enhanced = cv2.addWeighted(diff, 0.8, clahe.apply(s2), 0.2, 0)
    gray_enhanced = cv2.GaussianBlur(
        cv2.addWeighted(gray_enhanced, 0.85, clahe.apply(gray_raw), 0.15, 0),
        (3, 3), 2
    )

    rgb = cv2.cvtColor(boosted, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return np.concatenate([
        rgb,
        (diff.astype(np.float32) / 255.0)[:, :, None],
        (gray_enhanced.astype(np.float32) / 255.0)[:, :, None],
    ], axis=2)   # (H, W, 5)


# %%
# ============================================================
# DATA LOADING
# ============================================================

def load_data():
    """
    Load unified_dataset.json.
    Dataset 2 is skipped entirely (images invalid).
    Dots are already excluded from dataset 4 by build_dataset_json.py.
    Returns:
      aux_data  — list of dicts for auxiliary datasets (1, 3)
      orig_data — list of dicts for original dataset (4)
    """
    with open(UNIFIED_JSON) as f:
        entries = json.load(f)

    aux_data, orig_data = [], []

    for e in entries:
        if e["dataset"] == SKIP_DS_ID:
            continue
        if e["dataset"] == ORIGINAL_DS_ID:
            orig_data.append({"image_path": e["image_path"], "total_balls": e["total_balls"]})
        else:
            aux_data.append({"image_path": e["image_path"], "total_balls": e["total_balls"]})

    print(f"Auxiliary data : {len(aux_data)} images")
    print(f"Original data  : {len(orig_data)} images")
    arr = [d["total_balls"] for d in orig_data]
    print(f"  Ball count   : min={min(arr)}  max={max(arr)}  mean={np.mean(arr):.1f}  std={np.std(arr):.1f}")

    return aux_data, orig_data


aux_data, orig_data = load_data()


# %%
# ============================================================
# MASK VISUALIZATION
# Saves green/blue overlays to ./mascaras/ for all datasets
# except dataset 2 (already excluded in load_data).
# Green = inside mask, blue tint = outside mask.
# ============================================================

def save_mask_overlay(image_path, dataset_id, idx):
    img = cv2.imread(image_path)
    if img is None:
        print(f"  [skip] cannot read: {image_path}")
        return

    mask, _ = segment_table(img)

    overlay = img.copy().astype(np.float32)

    inside = mask == 255
    overlay[inside] = overlay[inside] * 0.5 + np.array([0, 180, 0], dtype=np.float32) * 0.5

    outside = mask == 0
    overlay[outside] = overlay[outside] * 0.5 + np.array([180, 0, 0], dtype=np.float32) * 0.5

    overlay = overlay.astype(np.uint8)

    # Use original filename from the path
    original_name = os.path.splitext(os.path.basename(image_path))[0]
    label = f"ds{dataset_id}_{original_name}"

    cv2.putText(overlay, label, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)

    cv2.imwrite(os.path.join(MASKS_DIR, f"{label}.jpg"), overlay)
    
def segment_table(img):
    """Detect the billiard table cloth and return (mask, corners).
    Iterates contours by area and picks the first one with uniform color
    (low H std inside the contour), rejecting banners and backgrounds.
    """
    h_img, w_img = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    blue_mask = cv2.inRange(hsv, np.array([90, 80, 80]), np.array([130, 255, 255]))
    green_mask = cv2.inRange(hsv, np.array([35,  80,  50]), np.array([ 95, 255, 255]))
    mask = cv2.bitwise_or(blue_mask, green_mask)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        full    = np.ones(img.shape[:2], dtype=np.uint8) * 255
        corners = np.array([[0, 0], [w_img, 0], [w_img, h_img], [0, h_img]], dtype=np.float32)
        return full, corners

    # Sort contours by area descending
    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    h_channel = hsv[:, :, 0]

    for contour in contours[:5]:  # check top 5 largest only
        epsilon = 0.02 * cv2.arcLength(contour, True)
        approx  = cv2.approxPolyDP(contour, epsilon, True)

        if len(approx) == 4:
            corners = approx.reshape(4, 2).astype(np.float32)
        else:
            rect    = cv2.minAreaRect(contour)
            corners = cv2.boxPoints(rect).astype(np.float32)

        # Build candidate mask and check H uniformity
        candidate_mask = np.zeros(img.shape[:2], dtype=np.uint8)
        cv2.fillPoly(candidate_mask, [corners.astype(np.int32)], 255)

        pixels = h_channel[candidate_mask == 255]
        if len(pixels) == 0:
            continue

        h_std = pixels.std()
        if h_std < 40:  # uniform color — likely felt
            corners    = order_corners(corners)
            clean_mask = np.zeros(img.shape[:2], dtype=np.uint8)
            cv2.fillPoly(clean_mask, [corners.astype(np.int32)], 255)
            return clean_mask, corners

    # Fallback — no uniform contour found, use largest
    contour = contours[0]
    epsilon = 0.02 * cv2.arcLength(contour, True)
    approx  = cv2.approxPolyDP(contour, epsilon, True)
    if len(approx) == 4:
        corners = approx.reshape(4, 2).astype(np.float32)
    else:
        rect    = cv2.minAreaRect(contour)
        corners = cv2.boxPoints(rect).astype(np.float32)

    corners    = order_corners(corners)
    clean_mask = np.zeros(img.shape[:2], dtype=np.uint8)
    cv2.fillPoly(clean_mask, [corners.astype(np.int32)], 255)
    return clean_mask, corners

def generate_mask_overlays():
    with open(UNIFIED_JSON) as f:
        entries = json.load(f)

    for idx, e in enumerate(entries):
        if e["dataset"] == SKIP_DS_ID:
            continue
        save_mask_overlay(e["image_path"], e["dataset"], idx)

generate_mask_overlays()


# %%
# ============================================================
# DATASETS AND DATALOADERS
# ============================================================

_train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
_val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
_MEAN_5CH = torch.tensor([0.485, 0.456, 0.406, 0.5, 0.5]).view(5, 1, 1)
_STD_5CH  = torch.tensor([0.229, 0.224, 0.225, 0.5, 0.5]).view(5, 1, 1)


class BallDataset(Dataset):
    def __init__(self, samples, augment=False):
        self.samples = samples
        self.tf      = _train_tf if augment else _val_tf

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s    = self.samples[idx]
        img  = cv2.imread(s["image_path"])
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(apply_table_mask(img), cv2.COLOR_BGR2RGB)
        return self.tf(Image.fromarray(img)), torch.tensor(float(s["total_balls"]))


class BallDataset5Ch(Dataset):
    def __init__(self, samples, augment=False):
        self.samples = samples
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        img = cv2.imread(s["image_path"])
        t   = (build_5channel(img) if img is not None
               else np.zeros((IMG_SIZE, IMG_SIZE, 5), dtype=np.float32))
        t   = cv2.resize(t, (IMG_SIZE, IMG_SIZE))

        if self.augment:
            if np.random.rand() > 0.5:
                t = np.fliplr(t).copy()
            if np.random.rand() > 0.5:
                t = np.flipud(t).copy()

        tensor = (torch.from_numpy(t.transpose(2, 0, 1)) - _MEAN_5CH) / _STD_5CH
        return tensor, torch.tensor(float(s["total_balls"]))


def make_loader(samples, augment=False, five_channel=False, bs=BATCH_SIZE):
    cls = BallDataset5Ch if five_channel else BallDataset
    return DataLoader(cls(samples, augment=augment),
                      batch_size=bs, shuffle=augment, num_workers=0, pin_memory=True)


# %%
# ============================================================
# MODEL DEFINITIONS
# ============================================================

def _reg_head(in_features, softplus=True):
    layers = [
        nn.Linear(in_features, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(128, 32), nn.ReLU(),
        nn.Linear(32, 1),
    ]
    if softplus:
        layers.append(nn.Softplus())
    return nn.Sequential(*layers)


class EfficientNetRegressor(nn.Module):
    """EfficientNet-B0 with regression head — MSE loss."""
    def __init__(self):
        super().__init__()
        bb               = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_f             = bb.classifier[1].in_features
        bb.classifier    = nn.Identity()
        self.backbone    = bb
        self.head        = _reg_head(in_f, softplus=False)

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(1)


class MobileNetFrozenRegressor(nn.Module):
    """MobileNetV3-Small fully frozen backbone — Poisson loss."""
    def __init__(self):
        super().__init__()
        bb = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        for p in bb.parameters():
            p.requires_grad = False
        self.features = bb.features
        self.avgpool  = bb.avgpool
        self.head     = _reg_head(576)

    def forward(self, x):
        with torch.no_grad():
            x = self.avgpool(self.features(x)).flatten(1)
        return self.head(x).squeeze(1)


class MobileNet5ChRegressor(nn.Module):
    """MobileNetV3-Small with 5-channel input, last 2 backbone layers trainable — Poisson loss."""
    def __init__(self):
        super().__init__()
        bb      = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        old     = bb.features[0][0]
        new     = nn.Conv2d(5, old.out_channels, old.kernel_size,
                            stride=old.stride, padding=old.padding, bias=False)
        with torch.no_grad():
            new.weight[:, :3] = old.weight
            new.weight[:, 3:] = old.weight[:, :2].mean(dim=1, keepdim=True)
        bb.features[0][0] = new

        for layer in list(bb.features.children())[:-2]:
            for p in layer.parameters():
                p.requires_grad = False

        self.features = bb.features
        self.avgpool  = bb.avgpool
        self.head     = _reg_head(576)

    def forward(self, x):
        return self.head(self.avgpool(self.features(x)).flatten(1)).squeeze(1)


class DinoRegressor(nn.Module):
    """DINOv2 backbone + MLP head. Supports 0-N unfrozen transformer blocks — Poisson loss."""
    def __init__(self, backbone_name="dinov2_vits14", unfreeze_blocks=0):
        super().__init__()
        self.backbone = torch.hub.load(
            "facebookresearch/dinov2", backbone_name, verbose=False
        )
        for p in self.backbone.parameters():
            p.requires_grad = False

        for block in self.backbone.blocks[-unfreeze_blocks:] if unfreeze_blocks > 0 else []:
            for p in block.parameters():
                p.requires_grad = True
        if unfreeze_blocks > 0:
            for p in self.backbone.norm.parameters():
                p.requires_grad = True

        self.head = _reg_head(self.backbone.embed_dim)

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(1)


# ── Model registry ────────────────────────────────────────────────────────────
MODEL_CONFIGS = [
    {"name": "dino_vits14_frozen",  "build": lambda: DinoRegressor("dinov2_vits14", 0), "loss": "poisson", "5ch": False},
    {"name": "dino_vits14_2blocks", "build": lambda: DinoRegressor("dinov2_vits14", 2), "loss": "poisson", "5ch": False},
    {"name": "dino_vitb14_2blocks", "build": lambda: DinoRegressor("dinov2_vitb14", 2), "loss": "poisson", "5ch": False},
]

# %%
# ============================================================
# TRAINING UTILITIES
# ============================================================

def get_criterion(loss_type):
    if loss_type == "mse":
        return nn.MSELoss()
    return nn.PoissonNLLLoss(log_input=False, full=True, reduction="mean")


def get_optimizer(model, lr, wd):
    """
    Differential LRs: head at lr, trainable backbone layers at lr/10.
    """
    head_ids        = {id(p) for p in model.head.parameters()} if hasattr(model, "head") else set()
    backbone_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    head_params     = [p for p in model.parameters() if p.requires_grad and id(p) in head_ids]

    groups = [{"params": head_params, "lr": lr}]
    if backbone_params:
        groups.append({"params": backbone_params, "lr": lr * 0.1})

    return torch.optim.AdamW(groups, weight_decay=wd)


def train_one_epoch(model, loader, optimizer, criterion, device, desc="train"):
    model.train()
    total = 0.0
    bar   = tqdm(loader, desc=desc, leave=False, ncols=80)
    for imgs, labels in bar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(imgs)
        bar.set_postfix(loss=f"{loss.item():.4f}")
    return total / len(loader.dataset)


@torch.no_grad()
def eval_epoch(model, loader, device, desc="val"):
    model.eval()
    preds_all, targets_all = [], []
    for imgs, labels in tqdm(loader, desc=desc, leave=False, ncols=80):
        preds_all.extend(model(imgs.to(device)).cpu().numpy())
        targets_all.extend(labels.numpy())
    p = np.array(preds_all,   dtype=np.float32)
    t = np.array(targets_all, dtype=np.float32)
    return float(np.mean(np.abs(p - t))), float(np.mean((p - t) ** 2))


def train_model(model, train_dl, val_dl, criterion, optimizer,
                n_epochs, patience, device, tag=""):
    """
    Train with ReduceLROnPlateau + early stopping.
    Returns (model with best weights loaded, best_mae).
    """
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=patience // 2, factor=0.5
    )
    best_mae   = float("inf")
    best_state = None
    no_improve = 0

    epoch_bar = tqdm(range(1, n_epochs + 1), desc=f"{tag}epochs", ncols=90)
    for epoch in epoch_bar:
        tl          = train_one_epoch(model, train_dl, optimizer, criterion, device, desc="  batch")
        val_mae, _  = eval_epoch(model, val_dl, device, desc="  val")
        scheduler.step(val_mae)

        if val_mae < best_mae:
            best_mae   = val_mae
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        epoch_bar.set_postfix(val_mae=f"{val_mae:.3f}", best=f"{best_mae:.3f}")

        if no_improve >= patience:
            tqdm.write(f"  {tag}Early stop epoch {epoch} — best MAE={best_mae:.3f}")
            break

    model.load_state_dict(best_state)
    return model, best_mae


# %%
# ============================================================
# PHASE 1 — PRE-TRAINING ON AUXILIARY DATASETS
# ============================================================

def pretrain(cfg, aux_data, device):
    name = cfg["name"]
    print(f"\n{'='*55}\n[Pre-train] {name}\n{'='*55}")

    rng      = np.random.RandomState(SEED)
    idx      = rng.permutation(len(aux_data))
    val_n    = max(1, int(0.1 * len(aux_data)))
    train_s  = [aux_data[i] for i in idx[val_n:]]
    val_s    = [aux_data[i] for i in idx[:val_n]]

    model     = cfg["build"]().to(device)
    criterion = get_criterion(cfg["loss"])
    optimizer = get_optimizer(model, PRETRAIN_LR, PRETRAIN_WD)

    train_dl = make_loader(train_s, augment=True,  five_channel=cfg["5ch"])
    val_dl   = make_loader(val_s,   augment=False, five_channel=cfg["5ch"])

    model, best_mae = train_model(
        model, train_dl, val_dl, criterion, optimizer,
        PRETRAIN_EPOCHS, PRETRAIN_PATIENCE, device, tag="[pretrain] "
    )

    save_path = os.path.join(MODELS_DIR, f"{name}_pretrained.pth")
    torch.save(model.state_dict(), save_path)
    print(f"  Saved pre-trained weights -> {save_path}  (Val MAE={best_mae:.3f})")

    return model.state_dict()


pretrained_states = {}
for cfg in tqdm(MODEL_CONFIGS, desc="pre-training models", ncols=90):
    pretrained_states[cfg["name"]] = pretrain(cfg, aux_data, device)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# %%
# ============================================================
# PHASE 2 — 5-FOLD CV ON ORIGINAL DATASET
# ============================================================

def run_kfold(cfg, orig_data, pretrained_state, device):
    name = cfg["name"]
    print(f"\n{'='*55}\n[5-Fold CV] {name}\n{'='*55}")

    indices  = np.random.RandomState(SEED).permutation(len(orig_data))
    folds    = np.array_split(indices, N_FOLDS)
    results  = []

    for fold_idx, test_idx in enumerate(folds):
        train_idx = np.concatenate([f for i, f in enumerate(folds) if i != fold_idx])
        train_s   = [orig_data[i] for i in train_idx]
        test_s    = [orig_data[i] for i in test_idx]

        val_n   = max(1, int(0.1 * len(train_s)))
        val_s   = train_s[-val_n:]
        train_s = train_s[:-val_n]

        model = cfg["build"]().to(device)
        model.load_state_dict(pretrained_state, strict=False)

        criterion = get_criterion(cfg["loss"])
        optimizer = get_optimizer(model, FINETUNE_LR, FINETUNE_WD)

        train_dl = make_loader(train_s, augment=True,  five_channel=cfg["5ch"])
        val_dl   = make_loader(val_s,   augment=False, five_channel=cfg["5ch"])
        test_dl  = make_loader(test_s,  augment=False, five_channel=cfg["5ch"])

        model, _ = train_model(
            model, train_dl, val_dl, criterion, optimizer,
            FINETUNE_EPOCHS, FINETUNE_PATIENCE, device,
            tag=f"[fold {fold_idx+1}] "
        )

        mae, mse = eval_epoch(model, test_dl, device)
        results.append({"mae": mae, "mse": mse})
        print(f"  Fold {fold_idx+1}/{N_FOLDS} -> MAE={mae:.3f}  MSE={mse:.3f}")

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    maes = [r["mae"] for r in results]
    mses = [r["mse"] for r in results]
    print(f"\n  CV Summary | MAE: {np.mean(maes):.3f} +/- {np.std(maes):.3f} | "
          f"MSE: {np.mean(mses):.3f} +/- {np.std(mses):.3f}")

    return results


cv_results = {}
for cfg in MODEL_CONFIGS:
    cv_results[cfg["name"]] = run_kfold(
        cfg, orig_data, pretrained_states[cfg["name"]], device
    )
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# %%
# ============================================================
# PHASE 3 — FINAL TRAINING ON ALL ORIGINAL DATA
# ============================================================

def final_train(cfg, orig_data, pretrained_state, device):
    name = cfg["name"]
    print(f"\n{'='*55}\n[Final Train] {name}\n{'='*55}")

    val_n   = max(1, int(FINAL_VAL_SPLIT * len(orig_data)))
    val_s   = orig_data[-val_n:]
    train_s = orig_data[:-val_n]

    model = cfg["build"]().to(device)
    model.load_state_dict(pretrained_state, strict=False)

    criterion = get_criterion(cfg["loss"])
    optimizer = get_optimizer(model, FINETUNE_LR, FINETUNE_WD)

    train_dl = make_loader(train_s, augment=True,  five_channel=cfg["5ch"])
    val_dl   = make_loader(val_s,   augment=False, five_channel=cfg["5ch"])

    model, best_mae = train_model(
        model, train_dl, val_dl, criterion, optimizer,
        FINETUNE_EPOCHS, FINETUNE_PATIENCE, device, tag="[final] "
    )

    save_path = os.path.join(MODELS_DIR, f"{name}_final.pth")
    torch.save(model.state_dict(), save_path)
    print(f"  Saved -> {save_path}  (Val MAE={best_mae:.3f})")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return best_mae


final_maes = {}
for cfg in MODEL_CONFIGS:
    final_maes[cfg["name"]] = final_train(
        cfg, orig_data, pretrained_states[cfg["name"]], device
    )


# %%
# ============================================================
# RESULTS SUMMARY
# ============================================================

header = (
    f"\n{'='*85}\n"
    f"TASK 2 — BALL COUNT REGRESSION  ({N_FOLDS}-fold CV)\n"
    f"{'='*85}\n"
    f"{'Model':<25} | {'CV MAE mean+/-std':<20} | {'CV MSE mean+/-std':<20} | Final Val MAE\n"
    f"{'-'*85}"
)

lines = [header]
for cfg in MODEL_CONFIGS:
    n    = cfg["name"]
    maes = [r["mae"] for r in cv_results[n]]
    mses = [r["mse"] for r in cv_results[n]]
    lines.append(
        f"{n:<25} | "
        f"{np.mean(maes):.3f} +/- {np.std(maes):.3f}        | "
        f"{np.mean(mses):.3f} +/- {np.std(mses):.3f}        | "
        f"{final_maes[n]:.3f}"
    )

lines.append("=" * 85)
summary = "\n".join(lines)
print(summary)

with open(RESULTS_FILE, "w") as f:
    f.write(summary)
print(f"\nResults saved -> {RESULTS_FILE}")

# ── Bar chart — CV MAE comparison ─────────────────────────────────────────────
short = [c["name"].replace("dino_", "").replace("mobilenet_", "mob_")
         .replace("efficientnet_", "eff_")
         for c in MODEL_CONFIGS]
means = [np.mean([r["mae"] for r in cv_results[c["name"]]]) for c in MODEL_CONFIGS]
stds  = [np.std( [r["mae"] for r in cv_results[c["name"]]]) for c in MODEL_CONFIGS]

plt.figure(figsize=(12, 5))
plt.bar(short, means, yerr=stds, capsize=6, color="steelblue", alpha=0.85)
plt.ylabel("MAE (balls)")
plt.title(f"Task 2 — {N_FOLDS}-Fold CV MAE by Model")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("task2_cv_comparison.png", dpi=150)
print("Plot saved -> task2_cv_comparison.png")

Device: cpu
Auxiliary data : 919 images
Original data  : 247 images
  Ball count   : min=2  max=16  mean=11.4  std=3.7


pre-training models:   0%|                                          | 0/3 [00:00<?, ?it/s]


[Pre-train] dino_vits14_frozen


KeyboardInterrupt: 

In [ ]:
# %%
"""
task2_benchmark.py — Ball Count Regression Benchmark

Transfer learning flow per model:
  1. Pre-train on auxiliary datasets (1, 3)
  2. Fine-tune on dataset 4 train split, early stopping on val split
  3. Evaluate on dataset 4 test split

Models:
  1. DINOv2 ViT-S/14 frozen + MLP     Poisson loss
  2. DINOv2 ViT-S/14 2 unfrozen       Poisson loss
  3. DINOv2 ViT-B/14 2 unfrozen       Poisson loss
"""

# %%
# ============================================================
# IMPORTS
# ============================================================
import json
import os
import copy
import warnings
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from tqdm import tqdm
warnings.filterwarnings("ignore")

# %%
# ============================================================
# GLOBAL CONFIG
# ============================================================

# --- paths ---
UNIFIED_JSON   = "unified_dataset.json"
MODELS_DIR     = "models"
RESULTS_FILE   = "task2_results.txt"
MASKS_DIR      = "mascaras"

# --- dataset ---
ORIGINAL_DS_ID = 4
SKIP_DS_ID     = 2

# --- model ---
IMG_SIZE           = 224
BATCH_SIZE         = 8
SEED               = 42

# --- pre-training (auxiliary datasets) ---
PRETRAIN_EPOCHS    = 30
PRETRAIN_LR        = 1e-3
PRETRAIN_WD        = 1e-2
PRETRAIN_PATIENCE  = 15

# --- fine-tuning (original dataset) ---
FINETUNE_EPOCHS    = 100
FINETUNE_LR        = 1e-3
FINETUNE_WD        = 1e-2
FINETUNE_PATIENCE  = 10

# ---- setup ----
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(MASKS_DIR,  exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# %%
# ============================================================
# TABLE SEGMENTATION + PREPROCESSING
# ============================================================
def order_corners(corners):
    """Order 4 corners as top-left, top-right, bottom-right, bottom-left."""
    corners = np.array(corners, dtype=np.float32)
    center  = corners.mean(axis=0)
    angles  = np.arctan2(corners[:, 1] - center[1], corners[:, 0] - center[0])
    order   = np.argsort(angles)
    corners = corners[order]
    top_idx = np.argmin(corners[:, 0] + corners[:, 1])
    corners = np.roll(corners, -top_idx, axis=0)
    return corners


def segment_table(img):
    """Detect the billiard table cloth and return (mask, corners).
    Iterates contours by area and picks the first one with uniform color
    (low H std inside the contour), rejecting banners and backgrounds.
    """
    h_img, w_img = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    blue_mask  = cv2.inRange(hsv, np.array([90, 80, 80]),  np.array([130, 255, 255]))
    green_mask = cv2.inRange(hsv, np.array([35, 80,  50]), np.array([ 95, 255, 255]))
    mask = cv2.bitwise_or(blue_mask, green_mask)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        full    = np.ones(img.shape[:2], dtype=np.uint8) * 255
        corners = np.array([[0, 0], [w_img, 0], [w_img, h_img], [0, h_img]], dtype=np.float32)
        return full, corners

    contours  = sorted(contours, key=cv2.contourArea, reverse=True)
    h_channel = hsv[:, :, 0]

    for contour in contours[:5]:
        epsilon = 0.02 * cv2.arcLength(contour, True)
        approx  = cv2.approxPolyDP(contour, epsilon, True)
        if len(approx) == 4:
            corners = approx.reshape(4, 2).astype(np.float32)
        else:
            rect    = cv2.minAreaRect(contour)
            corners = cv2.boxPoints(rect).astype(np.float32)

        candidate_mask = np.zeros(img.shape[:2], dtype=np.uint8)
        cv2.fillPoly(candidate_mask, [corners.astype(np.int32)], 255)
        pixels = h_channel[candidate_mask == 255]
        if len(pixels) == 0:
            continue
        if pixels.std() < 40:
            corners    = order_corners(corners)
            clean_mask = np.zeros(img.shape[:2], dtype=np.uint8)
            cv2.fillPoly(clean_mask, [corners.astype(np.int32)], 255)
            return clean_mask, corners

    # Fallback — use largest contour
    contour = contours[0]
    epsilon = 0.02 * cv2.arcLength(contour, True)
    approx  = cv2.approxPolyDP(contour, epsilon, True)
    if len(approx) == 4:
        corners = approx.reshape(4, 2).astype(np.float32)
    else:
        rect    = cv2.minAreaRect(contour)
        corners = cv2.boxPoints(rect).astype(np.float32)
    corners    = order_corners(corners)
    clean_mask = np.zeros(img.shape[:2], dtype=np.uint8)
    cv2.fillPoly(clean_mask, [corners.astype(np.int32)], 255)
    return clean_mask, corners


def apply_table_mask(img):
    mask, _ = segment_table(img)
    return cv2.bitwise_and(img, img, mask=mask)


def build_5channel(img):
    """
    Build (H, W, 5) float32 tensor:
      ch 0-2: RGB masked
      ch 3:   diff map (euclidean distance to cloth mean)
      ch 4:   gray_enhanced (diff + saturation CLAHE + gray)
    All channels normalized to [0, 1].
    """
    mask, _  = segment_table(img)
    masked   = cv2.bitwise_and(img, img, mask=mask)

    h, s, v  = cv2.split(cv2.cvtColor(masked, cv2.COLOR_BGR2HSV))
    boosted  = cv2.cvtColor(
        cv2.merge([h,
                   cv2.convertScaleAbs(s, alpha=1.35, beta=10),
                   cv2.convertScaleAbs(v, alpha=1.15, beta=8)]),
        cv2.COLOR_HSV2BGR
    )

    cloth_mean    = np.array(cv2.mean(boosted, mask=mask)[:3], dtype=np.float32)
    diff          = np.clip(
        np.sqrt(np.sum((boosted.astype(np.float32) - cloth_mean) ** 2, axis=2)),
        0, 255
    ).astype(np.uint8)

    _, s2, _      = cv2.split(cv2.cvtColor(boosted, cv2.COLOR_BGR2HSV))
    clahe         = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_raw      = cv2.cvtColor(boosted, cv2.COLOR_BGR2GRAY)
    gray_enhanced = cv2.addWeighted(diff, 0.8, clahe.apply(s2), 0.2, 0)
    gray_enhanced = cv2.GaussianBlur(
        cv2.addWeighted(gray_enhanced, 0.85, clahe.apply(gray_raw), 0.15, 0),
        (3, 3), 2
    )

    rgb = cv2.cvtColor(boosted, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return np.concatenate([
        rgb,
        (diff.astype(np.float32) / 255.0)[:, :, None],
        (gray_enhanced.astype(np.float32) / 255.0)[:, :, None],
    ], axis=2)


# %%
# ============================================================
# DATA LOADING
# ============================================================

def load_data():
    """
    Load unified_dataset.json.
    Dataset 2 is skipped entirely (images invalid).
    Returns:
      aux_data   — list of dicts for auxiliary datasets (1, 3)
      orig_train — dataset 4 train split
      orig_val   — dataset 4 val split
      orig_test  — dataset 4 test split
    """
    with open(UNIFIED_JSON) as f:
        entries = json.load(f)

    aux_data, orig_train, orig_val, orig_test = [], [], [], []

    for e in entries:
        if e["dataset"] == SKIP_DS_ID:
            continue
        item = {"image_path": e["image_path"], "total_balls": e["total_balls"]}
        if e["dataset"] != ORIGINAL_DS_ID:
            aux_data.append(item)
        else:
            spl = e.get("split", "train")
            if spl == "train":
                orig_train.append(item)
            elif spl == "val":
                orig_val.append(item)
            elif spl == "test":
                orig_test.append(item)

    print(f"Auxiliary data : {len(aux_data)} images")
    print(f"Original train : {len(orig_train)} images")
    print(f"Original val   : {len(orig_val)} images")
    print(f"Original test  : {len(orig_test)} images")

    return aux_data, orig_train, orig_val, orig_test


aux_data, orig_train, orig_val, orig_test = load_data()


# %%
# ============================================================
# MASK VISUALIZATION
# ============================================================

def save_mask_overlay(image_path, dataset_id, idx):
    img = cv2.imread(image_path)
    if img is None:
        print(f"  [skip] cannot read: {image_path}")
        return

    mask, _  = segment_table(img)
    overlay  = img.copy().astype(np.float32)
    inside   = mask == 255
    outside  = mask == 0
    overlay[inside]  = overlay[inside]  * 0.5 + np.array([0,   180, 0],   dtype=np.float32) * 0.5
    overlay[outside] = overlay[outside] * 0.5 + np.array([180, 0,   0],   dtype=np.float32) * 0.5
    overlay  = overlay.astype(np.uint8)

    original_name = os.path.splitext(os.path.basename(image_path))[0]
    label = f"ds{dataset_id}_{original_name}"
    cv2.putText(overlay, label, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.imwrite(os.path.join(MASKS_DIR, f"{label}.jpg"), overlay)


def generate_mask_overlays():
    with open(UNIFIED_JSON) as f:
        entries = json.load(f)
    for idx, e in enumerate(entries):
        if e["dataset"] == SKIP_DS_ID:
            continue
        save_mask_overlay(e["image_path"], e["dataset"], idx)

generate_mask_overlays()


# %%
# ============================================================
# DATASETS AND DATALOADERS
# ============================================================

_train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
_val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
_MEAN_5CH = torch.tensor([0.485, 0.456, 0.406, 0.5, 0.5]).view(5, 1, 1)
_STD_5CH  = torch.tensor([0.229, 0.224, 0.225, 0.5, 0.5]).view(5, 1, 1)


class BallDataset(Dataset):
    def __init__(self, samples, augment=False):
        self.samples = samples
        self.tf      = _train_tf if augment else _val_tf

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        img = cv2.imread(s["image_path"])
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(apply_table_mask(img), cv2.COLOR_BGR2RGB)
        return self.tf(Image.fromarray(img)), torch.tensor(float(s["total_balls"]))


class BallDataset5Ch(Dataset):
    def __init__(self, samples, augment=False):
        self.samples = samples
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        img = cv2.imread(s["image_path"])
        t   = (build_5channel(img) if img is not None
               else np.zeros((IMG_SIZE, IMG_SIZE, 5), dtype=np.float32))
        t   = cv2.resize(t, (IMG_SIZE, IMG_SIZE))
        if self.augment:
            if np.random.rand() > 0.5:
                t = np.fliplr(t).copy()
            if np.random.rand() > 0.5:
                t = np.flipud(t).copy()
        tensor = (torch.from_numpy(t.transpose(2, 0, 1)) - _MEAN_5CH) / _STD_5CH
        return tensor, torch.tensor(float(s["total_balls"]))


def make_loader(samples, augment=False, five_channel=False, bs=BATCH_SIZE):
    cls = BallDataset5Ch if five_channel else BallDataset
    return DataLoader(cls(samples, augment=augment),
                      batch_size=bs, shuffle=augment, num_workers=0, pin_memory=True)


# %%
# ============================================================
# MODEL DEFINITIONS
# ============================================================

def _reg_head(in_features, softplus=True):
    layers = [
        nn.Linear(in_features, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(128, 32), nn.ReLU(),
        nn.Linear(32, 1),
    ]
    if softplus:
        layers.append(nn.Softplus())
    return nn.Sequential(*layers)


class DinoRegressor(nn.Module):
    """DINOv2 backbone + MLP head. Supports 0-N unfrozen transformer blocks — Poisson loss."""
    def __init__(self, backbone_name="dinov2_vits14", unfreeze_blocks=0):
        super().__init__()
        self.backbone = torch.hub.load(
            "facebookresearch/dinov2", backbone_name, verbose=False
        )
        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in self.backbone.blocks[-unfreeze_blocks:] if unfreeze_blocks > 0 else []:
            for p in block.parameters():
                p.requires_grad = True
        if unfreeze_blocks > 0:
            for p in self.backbone.norm.parameters():
                p.requires_grad = True
        self.head = _reg_head(self.backbone.embed_dim)

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(1)


MODEL_CONFIGS = [
    {"name": "dino_vits14_frozen",  "build": lambda: DinoRegressor("dinov2_vits14", 0), "loss": "poisson", "5ch": False},
    {"name": "dino_vits14_2blocks", "build": lambda: DinoRegressor("dinov2_vits14", 2), "loss": "poisson", "5ch": False},
    {"name": "dino_vitb14_2blocks", "build": lambda: DinoRegressor("dinov2_vitb14", 2), "loss": "poisson", "5ch": False},
]

# %%
# ============================================================
# TRAINING UTILITIES
# ============================================================

def get_criterion(loss_type):
    if loss_type == "mse":
        return nn.MSELoss()
    return nn.PoissonNLLLoss(log_input=False, full=True, reduction="mean")


def get_optimizer(model, lr, wd):
    """Differential LRs: head at lr, trainable backbone layers at lr/10."""
    head_ids        = {id(p) for p in model.head.parameters()} if hasattr(model, "head") else set()
    backbone_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    head_params     = [p for p in model.parameters() if p.requires_grad and id(p) in head_ids]
    groups = [{"params": head_params, "lr": lr}]
    if backbone_params:
        groups.append({"params": backbone_params, "lr": lr * 0.1})
    return torch.optim.AdamW(groups, weight_decay=wd)


def train_one_epoch(model, loader, optimizer, criterion, device, desc="train"):
    model.train()
    total = 0.0
    bar   = tqdm(loader, desc=desc, leave=False, ncols=80)
    for imgs, labels in bar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(imgs)
        bar.set_postfix(loss=f"{loss.item():.4f}")
    return total / len(loader.dataset)


@torch.no_grad()
def eval_epoch(model, loader, device, desc="val"):
    model.eval()
    preds_all, targets_all = [], []
    for imgs, labels in tqdm(loader, desc=desc, leave=False, ncols=80):
        preds_all.extend(model(imgs.to(device)).cpu().numpy())
        targets_all.extend(labels.numpy())
    p = np.array(preds_all,   dtype=np.float32)
    t = np.array(targets_all, dtype=np.float32)
    return float(np.mean(np.abs(p - t))), float(np.mean((p - t) ** 2))


def train_model(model, train_dl, val_dl, criterion, optimizer,
                n_epochs, patience, device, tag=""):
    """Train with ReduceLROnPlateau + early stopping."""
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=patience // 2, factor=0.5
    )
    best_mae   = float("inf")
    best_state = None
    no_improve = 0

    epoch_bar = tqdm(range(1, n_epochs + 1), desc=f"{tag}epochs", ncols=90)
    for epoch in epoch_bar:
        train_one_epoch(model, train_dl, optimizer, criterion, device, desc="  batch")
        val_mae, _ = eval_epoch(model, val_dl, device, desc="  val")
        scheduler.step(val_mae)

        if val_mae < best_mae:
            best_mae   = val_mae
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        epoch_bar.set_postfix(val_mae=f"{val_mae:.3f}", best=f"{best_mae:.3f}")

        if no_improve >= patience:
            tqdm.write(f"  {tag}Early stop epoch {epoch} — best MAE={best_mae:.3f}")
            break

    model.load_state_dict(best_state)
    return model, best_mae


# %%
# ============================================================
# PHASE 1 — PRE-TRAINING ON AUXILIARY DATASETS
# ============================================================

def pretrain(cfg, aux_data, device):
    name = cfg["name"]
    print(f"\n{'='*55}\n[Pre-train] {name}\n{'='*55}")

    rng     = np.random.RandomState(SEED)
    idx     = rng.permutation(len(aux_data))
    val_n   = max(1, int(0.1 * len(aux_data)))
    train_s = [aux_data[i] for i in idx[val_n:]]
    val_s   = [aux_data[i] for i in idx[:val_n]]

    model     = cfg["build"]().to(device)
    criterion = get_criterion(cfg["loss"])
    optimizer = get_optimizer(model, PRETRAIN_LR, PRETRAIN_WD)

    train_dl = make_loader(train_s, augment=True,  five_channel=cfg["5ch"])
    val_dl   = make_loader(val_s,   augment=False, five_channel=cfg["5ch"])

    model, best_mae = train_model(
        model, train_dl, val_dl, criterion, optimizer,
        PRETRAIN_EPOCHS, PRETRAIN_PATIENCE, device, tag="[pretrain] "
    )

    save_path = os.path.join(MODELS_DIR, f"{name}_pretrained.pth")
    torch.save(model.state_dict(), save_path)
    print(f"  Saved pre-trained weights -> {save_path}  (Val MAE={best_mae:.3f})")
    return model.state_dict()


pretrained_states = {}
for cfg in tqdm(MODEL_CONFIGS, desc="pre-training models", ncols=90):
    pretrained_states[cfg["name"]] = pretrain(cfg, aux_data, device)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# %%
# ============================================================
# PHASE 2 — FINE-TUNE ON DATASET 4 TRAIN/VAL, EVALUATE ON TEST
# ============================================================

def finetune_and_eval(cfg, orig_train, orig_val, orig_test, pretrained_state, device):
    name = cfg["name"]
    print(f"\n{'='*55}\n[Fine-tune] {name}\n{'='*55}")

    model = cfg["build"]().to(device)
    model.load_state_dict(pretrained_state, strict=False)

    criterion = get_criterion(cfg["loss"])
    optimizer = get_optimizer(model, FINETUNE_LR, FINETUNE_WD)

    train_dl = make_loader(orig_train, augment=True,  five_channel=cfg["5ch"])
    val_dl   = make_loader(orig_val,   augment=False, five_channel=cfg["5ch"])
    test_dl  = make_loader(orig_test,  augment=False, five_channel=cfg["5ch"])

    model, best_val_mae = train_model(
        model, train_dl, val_dl, criterion, optimizer,
        FINETUNE_EPOCHS, FINETUNE_PATIENCE, device, tag="[finetune] "
    )

    test_mae, test_mse = eval_epoch(model, test_dl, device, desc="  test")
    print(f"  Val MAE={best_val_mae:.3f}  |  Test MAE={test_mae:.3f}  Test MSE={test_mse:.3f}")

    save_path = os.path.join(MODELS_DIR, f"{name}_final.pth")
    torch.save(model.state_dict(), save_path)
    print(f"  Saved -> {save_path}")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {"val_mae": best_val_mae, "test_mae": test_mae, "test_mse": test_mse}


results = {}
for cfg in MODEL_CONFIGS:
    results[cfg["name"]] = finetune_and_eval(
        cfg, orig_train, orig_val, orig_test,
        pretrained_states[cfg["name"]], device
    )
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# %%
# ============================================================
# RESULTS SUMMARY
# ============================================================

header = (
    f"\n{'='*75}\n"
    f"TASK 2 — BALL COUNT REGRESSION\n"
    f"{'='*75}\n"
    f"{'Model':<25} | {'Val MAE':>10} | {'Test MAE':>10} | {'Test MSE':>10}\n"
    f"{'-'*75}"
)

lines = [header]
for cfg in MODEL_CONFIGS:
    n = cfg["name"]
    r = results[n]
    lines.append(
        f"{n:<25} | {r['val_mae']:>10.3f} | {r['test_mae']:>10.3f} | {r['test_mse']:>10.3f}"
    )
lines.append("=" * 75)
summary = "\n".join(lines)
print(summary)

with open(RESULTS_FILE, "w") as f:
    f.write(summary)
print(f"\nResults saved -> {RESULTS_FILE}")

# Bar chart — Test MAE comparison
names  = [c["name"].replace("dino_", "") for c in MODEL_CONFIGS]
maes   = [results[c["name"]]["test_mae"] for c in MODEL_CONFIGS]

plt.figure(figsize=(8, 4))
plt.bar(names, maes, color="steelblue", alpha=0.85)
plt.ylabel("MAE (balls)")
plt.title("Task 2 — Test MAE by Model")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("task2_test_mae.png", dpi=150)
print("Plot saved -> task2_test_mae.png")